# 面向对象编程 · 知识点

### 1、类的定义与实例化

In [ ]:
# class 定义类，__init__ 是构造方法（不是构造器本身，是实例创建后第一个被调用的初始化方法）
# self 必须是实例方法的第一个参数，代表"当前实例"，等价于 Java 的 this，但 Python 要显式写出来
class Person:
    def __init__(self, name, age):
        self.name = name   # 实例属性，挂在 self 上
        self.age = age

p1 = Person("Tom", 18)
p2 = Person("Jerry", 20)
print(p1.name, p1.age)
print(p2.name, p2.age)

# 对比 Java：
# class Person {
#     String name; int age;
#     Person(String name, int age) { this.name = name; this.age = age; }
# }
# 区别：Python 不用声明字段类型，self 必须手写在参数列表里，Java 的 this 是隐式的

### 2、实例方法

In [ ]:
# 实例方法第一个参数也必须是 self，调用时 Python 自动把实例传进去，不用手动传
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def introduce(self):                 # 定义时写 self
        print(f"我是{self.name}，{self.age}岁")

    def have_birthday(self):
        self.age += 1                    # 通过 self 修改实例自己的属性

p = Person("Tom", 18)
p.introduce()        # 调用时不用传 self，p.introduce() 本质是 Person.introduce(p)
p.have_birthday()
p.introduce()

### 3、类属性 vs 实例属性

In [ ]:
# 类属性定义在类体里（不在 __init__ 里），所有实例共享同一份，类似 Java 的 static 字段
class Person:
    species = "Homo sapiens"   # 类属性，所有实例共享

    def __init__(self, name):
        self.name = name       # 实例属性，每个实例独立一份

p1 = Person("Tom")
p2 = Person("Jerry")
print(p1.species, p2.species)   # 都是 "Homo sapiens"，共享同一个

Person.species = "Changed"      # 通过类名修改类属性，影响所有实例
print(p1.species, p2.species)   # 都变了

# 常见用途：用类属性给所有实例计数
class Counter:
    count = 0                   # 类属性，记录总共创建了多少个实例

    def __init__(self):
        Counter.count += 1      # 通过类名修改，不要写 self.count += 1（那样会创建同名实例属性）

Counter(); Counter(); Counter()
print(Counter.count)   # 3

# 陷阱：类属性如果是可变对象（list/dict），所有实例会共享同一个，容易踩坑，
# 跟 04 章函数默认参数的陷阱是同一个道理
class Bad:
    items = []           # 危险！所有实例共享同一个列表
    def add(self, x):
        self.items.append(x)

### 4、类方法 & 静态方法

In [ ]:
# 实例方法第一个参数是 self（某个具体实例）
# 类方法用 @classmethod 装饰，第一个参数是 cls（类本身），常用来写"替代构造器"
# 静态方法用 @staticmethod 装饰，不接收 self 或 cls，本质就是放在类里的普通函数，跟实例/类都没关系
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    @classmethod
    def from_string(cls, s):
        # 替代构造器：从 "Tom-18" 这种字符串解析出一个 Person 实例
        name, age = s.split("-")
        return cls(name, int(age))   # cls(...) 等价于 Person(...)，但子类继承时会自动用子类

    @staticmethod
    def is_adult(age):
        # 跟具体某个实例无关的工具方法，只是逻辑上属于 Person
        return age >= 18

p = Person.from_string("Tom-18")
print(p.name, p.age)
print(Person.is_adult(20))

### 5、继承与方法重写

In [ ]:
# class Child(Parent) 继承，子类自动拥有父类的属性和方法
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        print(f"{self.name} 发出声音")

class Dog(Animal):
    def speak(self):              # 方法重写：同名方法覆盖父类的
        print(f"{self.name}: 汪汪")

class Cat(Animal):
    def __init__(self, name, color):
        super().__init__(name)    # 调用父类的 __init__，super() 拿到父类
        self.color = color

    def speak(self):
        print(f"{self.name}: 喵喵")

Dog("旺财").speak()
Cat("咪咪", "白色").speak()

# 对比 Java：class Dog extends Animal，super.xxx() 调父类方法/构造器，思路完全一致
# 区别：Python 没有 @Override 注解，重写就是单纯定义同名方法，不会做编译期检查

### 6、封装约定 & @property

In [ ]:
# Python 没有真正的 private，全靠命名约定：
# - 单下划线 _name：约定"内部使用，不建议外部访问"，但实际还是能访问
# - 双下划线 __name：触发"名字改写"（name mangling），变成 _类名__name，更难被外部直接访问到，但不是绝对私有
class Person:
    def __init__(self, name, age):
        self._name = name       # 约定内部字段
        self.__age = age        # 改写成 self._Person__age

p = Person("Tom", 18)
print(p._name)          # 能访问，只是不建议
# print(p.__age)        # 报错 AttributeError
print(p._Person__age)   # 但其实还是能通过改写后的名字访问到

# @property 把方法包装成"看起来像属性"，常用来做校验，对应 Java 的 getter/setter
class Person2:
    def __init__(self, age):
        self._age = age

    @property
    def age(self):              # getter，访问 p.age 不用加括号
        return self._age

    @age.setter
    def age(self, value):       # setter，赋值 p.age = x 时触发，可以加校验
        if value < 0:
            raise ValueError("年龄不能为负数")
        self._age = value

p2 = Person2(18)
print(p2.age)       # 18，不用写成 p2.age()
p2.age = 20         # 触发 setter
print(p2.age)
# p2.age = -1       # 触发 ValueError

### 7、魔术方法（dunder methods）

In [ ]:
# 魔术方法是 __xxx__ 形式的特殊方法，Python 在特定场景下自动调用它们
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        # print(obj) 或 f"{obj}" 时调用，给人看的友好字符串
        return f"Point({self.x}, {self.y})"

    def __repr__(self):
        # 直接在 REPL/调试器里显示对象、或者 print(列表) 时，调用的是 __repr__
        return f"Point(x={self.x}, y={self.y})"

    def __eq__(self, other):
        # 定义 == 的行为，不重写的话默认比较的是对象是否是同一个（id）
        return self.x == other.x and self.y == other.y

    def __add__(self, other):
        # 定义 + 的行为，让自定义对象支持运算符
        return Point(self.x + other.x, self.y + other.y)

    def __len__(self):
        # 定义 len(obj) 的行为
        return 2

p1 = Point(1, 2)
p2 = Point(1, 2)
print(p1)            # 调用 __str__ -> Point(1, 2)
print([p1])          # 列表里打印元素调用的是 __repr__
print(p1 == p2)       # True，用了自定义的 __eq__，不是默认的 id 比较
print(p1 + Point(3, 4))   # Point(4, 6)
print(len(p1))            # 2

### 8、多态与鸭子类型

In [ ]:
# Python 不需要显式声明"实现了某个接口"，只要对象有对应的方法/属性就能用，这叫"鸭子类型"
# （长得像鸭子、叫起来像鸭子，就当鸭子用，不关心它的真实类型）
class Circle:
    def __init__(self, r):
        self.r = r
    def area(self):
        return 3.14 * self.r ** 2

class Square:
    def __init__(self, side):
        self.side = side
    def area(self):
        return self.side ** 2

shapes = [Circle(2), Square(3)]
for shape in shapes:
    print(shape.area())   # 不用判断 shape 是什么类型，只要它有 area() 方法就能调用

# 对比 Java：Java 是静态类型，必须显式 implements 一个接口（如 Shape），
# 才能把不同类放进同一个 List<Shape> 里统一调用 area()；
# Python 不需要这层显式约束，只要"恰好都有 area() 方法"就行

### 9、抽象基类（ABC）

In [ ]:
# 鸭子类型灵活，但完全没有约束：如果子类忘了实现某个方法，要等运行到那一行才报错
# 如果想要类似 Java interface 那种"强制子类必须实现某些方法"的约束，用 abc 模块
from abc import ABC, abstractmethod

class Shape(ABC):              # 继承 ABC，表示这是一个抽象基类，不能被直接实例化
    @abstractmethod
    def area(self):             # 抽象方法，子类必须重写，否则不能实例化
        pass

class Circle(Shape):
    def __init__(self, r):
        self.r = r
    def area(self):
        return 3.14 * self.r ** 2

# shape = Shape()      # 报错：Can't instantiate abstract class Shape
c = Circle(2)
print(c.area())

class BadShape(Shape):
    pass   # 没有实现 area()

# bad = BadShape()     # 报错：Can't instantiate abstract class BadShape with abstract method area

### 10、dataclass：简化数据类

In [ ]:
# 如果一个类主要就是用来存数据（一堆属性 + 自动生成的 __init__/__repr__/__eq__），
# 手写很啰嗦，用 @dataclass 装饰器可以自动生成这些方法
from dataclasses import dataclass

@dataclass
class Point:
    x: int
    y: int

p1 = Point(1, 2)
p2 = Point(1, 2)
print(p1)            # 自动生成 __repr__: Point(x=1, y=2)
print(p1 == p2)       # 自动生成 __eq__: True（按字段值比较，不是默认的 id 比较）

# 等价于手写：
# class Point:
#     def __init__(self, x, y):
#         self.x = x
#         self.y = y
#     def __repr__(self):
#         return f"Point(x={self.x}, y={self.y})"
#     def __eq__(self, other):
#         return (self.x, self.y) == (other.x, other.y)

# 可以给字段设默认值，用法和普通参数默认值一样
@dataclass
class Config:
    debug: bool = False
    timeout: int = 30

print(Config())            # Config(debug=False, timeout=30)
print(Config(debug=True))  # Config(debug=True, timeout=30)